# Market-Oriented Reactive Power Sharing and Voltage Sensitivity Enhancement Operating Data Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cj0j-ft61/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
md = dataset.metadata
print(f"{md.name}: {md.description}\n\nVersion: {md.version} | Published: {md.datePublished}\nLicense: {md.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We use the `record_sets` property and list their `@id`s, field `@id`s, and column information.

In [ ]:
# List available record sets and their fields by @id
print('Available Record Sets:')
all_record_sets = list(dataset.record_sets)
for rs in all_record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name','')}")
    fields = rs.get('fields', []) or rs.get('field', []) or []
    if hasattr(fields, '__iter__') and not isinstance(fields, str):
        for field in fields:
            if isinstance(field, dict):
                print(f"   - Field @id: {field.get('@id', '')} name: {field.get('name','')}")
            else:
                print(f"   - Field @id: {field}")
    # Print columns
    for col in rs.get('columns', []) or rs.get('column', []) or []:
        print(f"   - Column @id: {col.get('@id','')} name: {col.get('name','')}")

### Example record from the first record set
Let's view a few records from one record set by its `@id`.

In [ ]:
# If there are any record sets, show a few records using their @id
if all_record_sets:
    example_record_set_id = all_record_sets[0]['@id']
    print(f"\nSample records from record set {example_record_set_id}:")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Gather all available record set @ids
record_set_ids = [r['@id'] for r in all_record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        recs = list(dataset.records(record_set=record_set_id))
        if recs:
            dataframes[record_set_id] = pd.DataFrame(recs)
            print(f"Loaded {len(recs)} records from Record Set {record_set_id}.")
        else:
            print(f"No records found for Record Set {record_set_id}.")
    except Exception as e:
        print(f"Error loading from {record_set_id}: {e}")

# For demonstration, select the first non-empty DataFrame
selected_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        selected_rs_id = rs_id
        break

if selected_rs_id:
    print(f"\nColumns in Record Set {selected_rs_id}:")
    print(dataframes[selected_rs_id].columns.tolist())
    display(dataframes[selected_rs_id].head())
else:
    print("No dataframes with records were loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data, and grouping by key attributes to prepare the data for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

df = dataframes[selected_rs_id] if selected_rs_id else pd.DataFrame()
if not df.empty:
    # Try to select a numeric field automatically
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        # Try to convert any columns to numeric if possible
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using field '{numeric_field}' for numeric EDA.")
        
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (threshold = mean): {len(filtered_df)} records")
        display(filtered_df.head())
        
        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
        # Available group fields
        possible_group_fields = [col for col in df.columns if col != numeric_field]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            if group_field in filtered_df.columns and not filtered_df[group_field].isnull().all():
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped data by '{group_field}' (mean of {numeric_field}):")
                display(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_fields:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # If grouping field exists and is categorical, show boxplot
    if possible_group_fields:
        group_field = possible_group_fields[0]
        if group_field in df.columns and (df[group_field].dtype == object or df[group_field].nunique() < 20):
            plt.figure(figsize=(8, 4))
            sns.boxplot(x=df[group_field], y=df[numeric_field])
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=90)
            plt.show()
else:
    print("No numeric data to plot.")

## 6. Conclusion
We loaded the FAIR^2 operating data for Nordic 44 and CIGRE 32 Power Systems using `mlcroissant`. We reviewed available record sets, explored numeric distributions, filtered and normalized values, and visualized the data.

For deeper analysis, continue with domain-specific feature engineering and modeling based on the power systems measurements and scenario data.